# Use case 2 — Codelist bindings: forward inventory and reverse lookup

**Questions:**

1. *Forward.* What CT bindings exist in USDM v4? How are they distributed
   across source categories (CDISC Library / SDTM CT / Protocol CT / external)?
2. *Reverse.* Given a codelist C-Code, which USDM classes carry it as a Code
   value — including via inheritance?

**Pinned to v0.1.** Reads `usdm:boundCodelist` (NCIt IRI) and
`usdm:boundCodelistNote` (raw `Has Value List` text from `USDM_CT.xlsx`).


In [1]:
import rdflib
import pandas as pd
import re
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
EXPECTED_BASELINE = 8173

if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/10/20 first to regenerate."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
n_triples = len(g)
print(f"parsed {n_triples} triples from {TTL_PATH}")
if n_triples != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.1 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.1 binding shape")

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]


parsed 8173 triples from ../usdm_v4.ttl


## Step 1 — forward inventory

Every property carrying a codelist binding annotation. The v0.1 layer
emits `usdm:boundCodelistNote` for every Y-row in `USDM_CT.xlsx`
(declaring class only) and additionally `usdm:boundCodelist` when the
cell text yields a structured C-code.


In [2]:
INVENTORY_SPARQL = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4#>

SELECT ?prop ?domain ?boundCodelist ?boundCodelistNote
WHERE {
  ?prop usdm:boundCodelistNote ?boundCodelistNote .
  ?prop rdfs:domain ?domain .
  OPTIONAL { ?prop usdm:boundCodelist ?boundCodelist }
}
"""

rows = []
for r in g.query(INVENTORY_SPARQL):
    prop_short = short(r["prop"])
    rows.append({
        "domain":            short(r["domain"]),
        "prop":              prop_short,
        "attr":              prop_short.split("-", 1)[-1],
        "boundCodelist":     short(r["boundCodelist"]),
        "boundCodelistNote": str(r["boundCodelistNote"]),
    })
bindings_df = pd.DataFrame(rows).sort_values(["domain", "attr"]).reset_index(drop=True)
print(f"{len(bindings_df)} declaring-class bindings")
print(f"  with structured codelist: {bindings_df['boundCodelist'].notna().sum()}")
print(f"  free-text only:           {bindings_df['boundCodelist'].isna().sum()}")
bindings_df.head(10)


57 declaring-class bindings
  with structured codelist: 45
  free-text only:           12


,domain,prop,attr,boundCodelist,boundCodelistNote
0,Address,Address-country,country,NaN,Y (Point out to ISO 3166-1 Alpha-3 Country code)
1,AdministrableProduct,AdministrableProduct-administrableDoseForm,administrableDoseForm,C66726,Y (SDTM Terminology Codelist C66726)
2,AdministrableProduct,AdministrableProduct-pharmacologicClass,pharmacologicClass,NaN,"Y (Points to external codelists such as UNII, ..."
3,AdministrableProduct,AdministrableProduct-productDesignation,productDesignation,C207418,Y (C207418)
4,AdministrableProduct,AdministrableProduct-sourcing,sourcing,C215483,Y (C215483)
5,AdministrableProductProperty,AdministrableProductProperty-type,type,C215479,Y (C215479)
6,Administration,Administration-frequency,frequency,C71113,Y (SDTM Terminology Codelist C71113)
7,Administration,Administration-route,route,C66729,Y (SDTM Terminology Codelist C66729)
8,EligibilityCriterion,EligibilityCriterion-category,category,C66797,Y (SDTM Terminology Codelist C66797)
9,Encounter,Encounter-contactModes,contactModes,C171445,Y (SDTM Terminology Codelist C171445)


## Step 2 — categorize by source

Parse the raw `boundCodelistNote` cell text into source categories:

- `cdisc_library` — bare `Y (Cxxxxxx)`, codelist lives in CDISC Library RDF
- `sdtm_ct`       — `Y (SDTM Terminology Codelist Cxxxxxx)`
- `protocol_ct`   — `Y (Protocol Terminology Codelist Cxxxxxx)`
- `external`      — free-text pointers to ISO 3166, ISO 639, MedDRA, SNOMEDCT, FHIR, etc.


In [3]:
def categorize_source(note):
    if "SDTM Terminology Codelist" in note:
        return "sdtm_ct"
    if "Protocol Terminology Codelist" in note:
        return "protocol_ct"
    if re.fullmatch(r"Y \(C\d+\)", note):
        return "cdisc_library"
    return "external"

bindings_df["source"] = bindings_df["boundCodelistNote"].apply(categorize_source)

breakdown = bindings_df.groupby("source").agg(
    total=("prop", "count"),
    structured=("boundCodelist", lambda s: int(s.notna().sum())),
    free_text=("boundCodelist", lambda s: int(s.isna().sum())),
).reset_index()
print("=== bindings by source ===")
print(breakdown.to_string(index=False))
print()

for src in ["cdisc_library", "sdtm_ct", "protocol_ct", "external"]:
    subset = bindings_df[bindings_df["source"] == src]
    if len(subset) == 0:
        continue
    print(f"--- {src} ({len(subset)}) — first 3 ---")
    print(subset[["domain", "attr", "boundCodelist", "boundCodelistNote"]].head(3).to_string(index=False))
    print()


=== bindings by source ===
       source  total  structured  free_text
cdisc_library     25          25          0
     external     12           0         12
  protocol_ct      1           1          0
      sdtm_ct     19          19          0

--- cdisc_library (25) — first 3 ---
                      domain               attr boundCodelist boundCodelistNote
        AdministrableProduct productDesignation       C207418       Y (C207418)
        AdministrableProduct           sourcing       C215483       Y (C215483)
AdministrableProductProperty               type       C215479       Y (C215479)

--- sdtm_ct (19) — first 3 ---
              domain                  attr boundCodelist                    boundCodelistNote
AdministrableProduct administrableDoseForm        C66726 Y (SDTM Terminology Codelist C66726)
      Administration             frequency        C71113 Y (SDTM Terminology Codelist C71113)
      Administration                 route        C66729 Y (SDTM Terminology Code

## Step 3 — reverse lookup, declaring-class only

Given a codelist C-Code, which property is bound to it? Demo: `C66732`,
SDTM CT *Sex*.


In [4]:
DEMO_CCODE = "C66732"  # SDTM CT Sex

def properties_for_codelist(ccode):
    """Return DataFrame of properties bound to the given NCIt codelist C-code (declaring class only)."""
    if not re.fullmatch(r"C\d+", ccode):
        raise ValueError(f"expected NCIt C-code form Cxxxxxx, got {ccode!r}")
    sparql = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX usdm: <https://w3id.org/cdisc/usdm/v4#>
    PREFIX ncit: <http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#>

    SELECT ?prop ?domain ?note
    WHERE {
      ?prop usdm:boundCodelist ncit:%s ;
            rdfs:domain ?domain ;
            usdm:boundCodelistNote ?note .
    }
    """ % ccode
    rows = []
    for r in g.query(sparql):
        prop_short = short(r["prop"])
        rows.append({
            "domain": short(r["domain"]),
            "prop":   prop_short,
            "attr":   prop_short.split("-", 1)[-1],
            "note":   str(r["note"]),
        })
    return pd.DataFrame(rows)

print(f"=== declaring-class binding for ncit:{DEMO_CCODE} ===")
declaring = properties_for_codelist(DEMO_CCODE)
declaring


=== declaring-class binding for ncit:C66732 ===


,domain,prop,attr,note
0,PopulationDefinition,PopulationDefinition-plannedSex,plannedSex,Y (SDTM Terminology Codelist C66732)


## Step 4 — reverse lookup, inheritance-aware

The useful version: which *classes* would carry this codelist as a Code
value, including all subclasses that inherit the bound property.

The SPARQL walks `rdfs:subClassOf*` from the declaring class — the `*`
is zero-or-more steps, so the declaring class itself is included.


In [5]:
def classes_using_codelist(ccode):
    """Return DataFrame of (declaring_class, effective_class, attr) using the codelist."""
    if not re.fullmatch(r"C\d+", ccode):
        raise ValueError(f"expected NCIt C-code form Cxxxxxx, got {ccode!r}")
    sparql = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX usdm: <https://w3id.org/cdisc/usdm/v4#>
    PREFIX ncit: <http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#>

    SELECT ?prop ?declaring ?effective
    WHERE {
      ?prop usdm:boundCodelist ncit:%s ;
            rdfs:domain ?declaring .
      ?effective rdfs:subClassOf* ?declaring .
      FILTER(isIRI(?effective))
    }
    """ % ccode
    rows = []
    for r in g.query(sparql):
        prop_short = short(r["prop"])
        rows.append({
            "declaring_class": short(r["declaring"]),
            "effective_class": short(r["effective"]),
            "attr":            prop_short.split("-", 1)[-1],
        })
    return pd.DataFrame(rows).sort_values(["declaring_class", "effective_class"]).reset_index(drop=True)

effective = classes_using_codelist(DEMO_CCODE)
print(f"=== effective classes carrying ncit:{DEMO_CCODE} ===")
print(f"{len(effective)} class(es), declaring + subclasses")
effective


=== effective classes carrying ncit:C66732 ===
3 class(es), declaring + subclasses


,declaring_class,effective_class,attr
0,PopulationDefinition,PopulationDefinition,plannedSex
1,PopulationDefinition,StudyCohort,plannedSex
2,PopulationDefinition,StudyDesignPopulation,plannedSex


## Step 5 — complete reverse index

For every distinct `usdm:boundCodelist` C-code in the graph, accumulate
the inheritance-aware reverse lookup into a single index. One row per
(codelist, declaring_class, effective_class, attr).

Exported as `reverse_codelist_index.csv` in the `examples/` directory.
Tracked in git — regenerated on every run.


In [6]:
distinct_ccodes = sorted(bindings_df["boundCodelist"].dropna().unique())
print(f"{len(distinct_ccodes)} distinct codelist C-codes in v0.1")

frames = []
for ccode in distinct_ccodes:
    df = classes_using_codelist(ccode)
    df.insert(0, "boundCodelist", ccode)
    frames.append(df)

reverse_idx_df = pd.concat(frames, ignore_index=True)

CSV_PATH = "reverse_codelist_index.csv"
reverse_idx_df.to_csv(CSV_PATH, index=False)

print(f"reverse index: {len(reverse_idx_df)} rows × {len(reverse_idx_df.columns)} columns")
print(f"columns: {list(reverse_idx_df.columns)}")
print(f"wrote {CSV_PATH}")
reverse_idx_df.head(15)


45 distinct codelist C-codes in v0.1
reverse index: 53 rows × 4 columns
columns: ['boundCodelist', 'declaring_class', 'effective_class', 'attr']
wrote reverse_codelist_index.csv


,boundCodelist,declaring_class,effective_class,attr
0,C127259,ObservationalStudyDesign,ObservationalStudyDesign,model
1,C127260,ObservationalStudyDesign,ObservationalStudyDesign,samplingMethod
2,C127261,ObservationalStudyDesign,ObservationalStudyDesign,timePerspective
3,C127262,Encounter,Encounter,environmentalSettings
4,C171445,Encounter,Encounter,contactModes
5,C174222,StudyArm,StudyArm,type
6,C188723,StudyDefinitionDocumentVersion,StudyDefinitionDocumentVersion,status
7,C188724,Organization,Organization,type
8,C188725,Objective,Objective,level
9,C188726,Endpoint,Endpoint,level


## Discussion

What this demonstrates:

- The forward inventory is a one-SPARQL view of the v0.1 binding layer.
  57 declaring-class bindings, source-categorized cleanly via the
  `boundCodelistNote` text.
- The reverse lookup answers "which classes need this codelist?"
  honestly including inheritance — the binding lives on the declaring
  property, but applies to every subclass through `rdfs:subClassOf*`.

What's not shown:

- **Permitted-term resolution.** `usdm:boundCodelist` gives the codelist
  C-Code; the actual permitted Code values live in CDISC Library RDF or
  the Library REST API. A future Library/CT extension closes that loop.
- **Free-text bindings cannot be searched by C-Code.** The 12 free-text
  bindings (ISO 3166, ISO 639, MedDRA, SNOMEDCT, etc.) appear in the
  forward inventory but not in the reverse index — by design, since
  there's no structured anchor to walk back from.

Pointer: case 3 takes on the gap analysis — Code-typed properties
without bindings, polymorphic union ranges, and other "what's not yet
bound?" views.
